Cipher Code.
This contains 6 ciphers which can encode, decode and be composed to create new ciphers. This is some text used for testing and examples alongside alphabets that can be singled out to be encoded seperately. Examples are at the end.


In [16]:
bond = 'Agent 007: "Shaken, not stirred!"'
holmes = '[At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"'
joker = "Why so serious?! Let's put a smile on that face."
turing = 'In 1950, Turing asked: "Can machines think?"'
hamlet = "[Act 3, Scene 1] To be, or not to be?"
enigma = "[Bletchley, 1941] The Enigma has been cracked!"
alp = ['a','b','c','d','e','f','g','h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','w','x','y','z']
vowels = ['a','e','i','o','u']
consonants = ['b','c','d','f','g','h','j','k','l','m','n','p','q','r','s','t','v','w','x','y','z']
qwerty = ['q','w','e','r','t','y','u','i','o','p','a','s','d','f','g','h','j','k','l','z','x','c','v','b','n','m']
ambidextrous = ['a','m','b','i','d','e','x','t','r','o','u','s']
texts_list = [bond,holmes,joker,turing,hamlet,enigma]
import math
import copy
import numpy as np

Cipher Class.
This is the fundamental class of ciphers to which each branch belongs to (besides affine-hill cipher). This has an encode, decode, compose and power function. These can be used with any custom alphabet to encodeany desired text.

In [2]:
class Cipher(object):
    def __init__(self,encoded_alphabet):
        self.enc_alp = encoded_alphabet
        self.enc_alp_cap =[c.upper() for c in encoded_alphabet] 
        self.base = ['a','b','c','d','e','f','g','h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','w','x','y','z']
        self.base_cap =[c.upper() for c in self.base] 
        
    def encode(self,text): # encode is  hardcoded so that each letter in the alphabet is mapped first and then each letter's index in the text is identified then taken from the corresponding point
        k = len(text)
        ctext = list(text)
        r = ""
        for j in range(k):
            for i in range(26):
                if ctext[j] == self.base[i]:
                    ctext[j] = self.enc_alp[i]
                    break
                elif ctext[j] == self.base_cap[i]:
                    ctext[j] = self.enc_alp_cap[i]
                    break
            r += ctext[j]     
        return r

    def decode(self,text):
        k = len(text)
        ctext = list(text)
        r = ""
        for j in range(k):
            for i in range(26):
                if ctext[j] == self.enc_alp[i]:
                    ctext[j] = self.base[i]
                    break
                    
                elif ctext[j] == self.enc_alp_cap[i]:
                    ctext[j] = self.base_cap[i]
                    break
            r += ctext[j]     
        return r

    def compose(self,f): ## compose self o f
        g = list(self.base)
        for j in range(26):
            for i in range(26):
                if f.enc_alp[j] == self.base[i]:
                    g[j] = self.enc_alp[i]
                    break
        return Cipher(g)

    def __pow__(self,n):
        if n == 0:
            return Cipher(self.base)
        else:
            g = self
            for j in range(n-1):
                g = g.compose(self)
            return g


Caeser Cipher.
This cipher will move each letter in the given alphabet a certain number of positions along. For example, a Caesar cipher with transformation 2 over the vowels will transform A to I but B will be unchanged. Only two arguments are required for a Caesar cipher, the transformation and the desired alphabet to change. With this code you can multiply, add and subtract to form new Caesar ciphers. 

In [3]:
class Caesar(Cipher):
    def __init__(self,n,alphabet):
        self.base = ['a','b','c','d','e','f','g','h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','w','x','y','z']
        self.alp = alphabet
        self.alp_cap =[c.upper() for c in alphabet]
        self.k = len(alphabet)
        self.n = n % self.k
        self.enc_alp = list(self.base)
        for j in range(self.k):
            for i in range(26):
                if self.alp[j] == self.base[i]:
                    self.enc_alp[i] = self.alp[(j+self.n)%self.k]
                    break
        self.enc_alp_cap =[c.upper() for c in self.enc_alp]
        super().__init__(self.enc_alp)

    def __str__(self): # describes transformation
        r = ""
        for j in range(self.k):
            r += self.alp[j]
        return "Caesar cipher with shift n = " + str(self.n) + " over " + str(r) 

    def __repr__(self):
        return str(self)


    def __add__(self,f): # addition of Caesar functions on same alphabet and same magnitude of a component
        if self.alp != f.alp or self.alp_cap != f.alp_cap:
            raise Exception("alphabets are not compatible")
        else:
            return Caesar((self.n + f.n)%self.k,self.alp)

    def __mul__(self,n):
            return Caesar((self.n*n)%self.k,self.alp)

    def __rmul__(self,n): # for both directions of multiplaction, multiplies shift component by n
        return Caesar((self.n*n)%self.k,self.alp)

    def __neg__(self): # makes the n component negative
        return Caesar((-self.n)%self.k,self.alp)

    def __sub__(self,f):
        return self + (-f)

Affine Cipher.
The affine cipher can complete a linear transformation given the components are integers. Each letter's position in the alphabet will be shifted aN + b. When the length of the given alphabet and a are not coprime this raises an exception when decoding as there are no unique solutions to the modular equation used to undo the transformation. In a regular alphabet using affine cipher with a = 3 and b = 3, the letter A is transformed (A has index 1 so 3x1+3 = 6) to become F.

In [4]:
class Affine(Cipher):
    def __init__(self,a,b,alphabet):
        self.base = ['a','b','c','d','e','f','g','h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','w','x','y','z']
        self.alp = alphabet
        self.alp_cap =[c.upper() for c in alphabet]
        self.k = len(alphabet)
        self.a = a
        self.b = b % self.k
        self.enc_alp = list(self.base)
        for j in range(self.k):
            for i in range(26):
                if self.alp[j] == self.base[i]:
                    self.enc_alp[i] = self.alp[(self.a*j+self.b)%self.k]
                    break
        self.enc_alp_cap =[c.upper() for c in self.enc_alp]
        super().__init__(self.enc_alp)

    def __str__(self): # describes transformation
        r = ""
        for j in range(self.k):
            r += self.alp[j]
        return "Affine cipher with transfomation a = " + str(self.a) + " and b = " + str(self.b) + " over " + str(r) 

    def __repr__(self):
        return str(self)


    def __add__(self,f): # addition of affine functions on same alphabet and same magnitude of a component
        if self.alp != f.alp or self.alp_cap != f.alp_cap:
            raise Exception("alphabets are not compatible")
        elif self.a != f.a:
            raise Exception("To add, multiplicative transformations must be the same magnitude")
        else:
            return Affine(self.a,(self.b + f.b)%self.k,self.alp)

    def __mul__(self,n):
            return Affine(self.a,(self.b*n)%self.k,self.alp)

    def __rmul__(self,n): # for both directions of multiplaction, multiplies b component by n
        return Affine(self.a,(self.b*n)%self.k,self.alp)

    def __neg__(self): # makes the b component negative
        return Affine(self.a,(-self.b)%self.k,self.alp)

    def __sub__(self,f):
        return self + (-f)
     
    def decode(self,text):
        if math.gcd(self.a,self.k) == 1:
            return super().decode(text)
        else:
            raise Exception("Since a and alphabet length are not coprime, there is no unique solution to modular equation. Decode function fails.") # adds exception when no real solutions

Reverse Cipher. The reverse cipher is arguably the most simple, it reverses the alphabet given and prints. For a regular alphabet, A becomes Z, B becomes Y. Only requires the alphabet component.

In [5]:
class Reverse(Cipher):
    def __init__(self,alphabet):
        self.alp = alphabet
        self.alp_cap =[c.upper() for c in alphabet]
        self.k = len(self.alp)
        self.base = ['a','b','c','d','e','f','g','h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','w','x','y','z']
        self.enc_alp = list(self.base)
        for j in range(self.k):
            for i in range(len(self.base)):
                if self.alp[j] == self.base[i]:
                    self.enc_alp[i] = self.alp[self.k - j - 1]
                    break
                    
        self.enc_alp_cap =[c.upper() for c in self.enc_alp]
        super().__init__(self.enc_alp)
        
    def __str__(self): # describes transformation
        r = ""
        for j in range(self.k):
            r += self.alp[j]
        return "Reverse Cipher over " + r

    def __repr__(self):
        return str(self)    


Split Cipher. The split cipher breaks the alphabet into evens and odds and places the evens then the reversed odds. On a regular alphabet this looks like acegikmoqsuwyzxvtrpnljhfdb (I used my own code to write this out). Only requires the alphabet component.

In [6]:
class Split(Cipher):
    def __init__(self,alphabet):
        self.alp = alphabet
        self.alp_cap =[c.upper() for c in alphabet]
        self.k = len(self.alp)
        self.base = ['a','b','c','d','e','f','g','h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','w','x','y','z']
        Evens = self.alp[:self.k:2]
        Odds = self.alp[1:self.k:2][::-1]
        Encoded = Evens + Odds
        self.enc_alp = list(self.base)
        for j in range(self.k):
            for i in range(len(self.base)):
                if self.alp[j] == self.base[i]:
                    self.enc_alp[i] = Encoded[j]
                    break
        self.enc_alp_cap =[c.upper() for c in self.enc_alp]
        super().__init__(self.enc_alp)
        
    def __str__(self): # describes transformation
        r = ""
        for j in range(self.k):
            r += self.alp[j]
        return "Split Cipher over " + r

    def __repr__(self):
        return str(self)   

Affine Hill Cipher. This requires its own unique class because it doesn't affect alphabets the same way regular ciphers do. This applies the transformation of a square matrix to the index of each letter. For example, over a regular alphabet when you have a 3x3 matrix M = ((1,0,1),(0,1,0),(0,0,1)) and you encode the string "abc", the affine cipher will complete M o (1,2,3) = (4,2,3) and return "dbc". Any spare letters in the string say if a string was 4 characters long, the last letter is untouched. This can only be decoded if the matrix has an inverse and the determinant of the matrix is coprime to the chose alphabet. Alongside this, a vector b can be added to the index of the letters in the string, like a caesar cipher. These ciphers can be composed with eachother for a specific text but will not create a unique Affine Hill Cipher since this allows different size matricies to be composed. 

In [7]:
class Affine_Hill_Cipher(object):
    def __init__(self,A,b,alphabet):
        self.n = int(np.sqrt(np.size(A)))
        self.A = A
        self.b = b
        self.alp = alphabet
        self.alp_cap = [c.upper() for c in alphabet]
        self.k = len(self.alp)

    def encode(self,text):
        ctext = list(text)
        k = len(ctext)
        vec = []
        index_list = []
        enc_list = []
        pos = []
        cap_pos = []
        r = ""
        for j in range(k):
            for i in range(self.k):
                if ctext[j] == self.alp[i]:
                    index_list.append(i)
                    pos.append(j)
                elif ctext[j] == self.alp_cap[i]:
                    index_list.append(i)
                    pos.append(j)
                    cap_pos.append(j) # position and capital_position for differentiation later
        n = len(pos)
        for i in range(n//self.n): # to only include as many vectors as can fit and ignore final few values that wont.
            for j in range(self.n):
                vec.append(index_list[self.n*i+j]) # splits index list into many vectors to be encoded
            vector = np.array(vec)
            encoded_vector = ((self.A@vector) + self.b) % self.k # encodes and returns values less than length of alphabet
            for m in range(self.n):
                enc_list.append(int(encoded_vector[m])) # combines them into a large list
            vec = [] # resets vector so it can be reused
            
        for ell in range(len(enc_list)): # only up to length of enc_list to avoid overwritting remainder letters
            if pos[ell] in cap_pos:
                ctext[pos[ell]] = self.alp_cap[enc_list[ell]]
            else:
                ctext[pos[ell]] = self.alp[enc_list[ell]] # converts each letter to its encoded one using the position and encoded index from lists formed
        for j in range(k):
            r += ctext[j] # constructs final string
        return r 

    def decode(self,text): # copies encoded
        ctext = list(text)
        k = len(ctext)
        vec = []
        index_list = []
        dec_list = []
        pos = []
        cap_pos = []
        r = ""
        if np.linalg.det(A) == 0:
            raise Exception("No inverse of A so there is no unique decoded string.")
        elif math.gcd(int(np.linalg.det(A)),self.k) != 1:
            raise Exception("No det(A) and alphabet length are not coprime so decoding is not unique") # raises exceptions for invalid or non-coprime determinants 
        det = round(np.linalg.det(self.A)) # rounded det(A) to prevent floats' innacurracy affecting results.
        inverse = np.linalg.inv(self.A) * pow(int(det),-1,self.k) * int(det) # constructs inverse A
        
        for j in range(k):
            for i in range(self.k):
                if ctext[j] == self.alp[i]:
                    index_list.append(i)
                    pos.append(j)
                elif ctext[j] == self.alp_cap[i]:
                    index_list.append(i)
                    pos.append(j)
                    cap_pos.append(j)
        n = len(pos)
        for i in range(n//self.n):
            for j in range(self.n):
                vec.append(index_list[self.n*i+j])
            vector = np.array(vec)
            decoded_vector = (inverse@(vector - self.b)) % self.k # new equation using inverse A
            for m in range(self.n):
                dec_list.append(int(decoded_vector[m])) # encoded list is now decoded list :)
            vec = []
            
        for ell in range(len(dec_list)):
            if pos[ell] in cap_pos:
                ctext[pos[ell]] = self.alp_cap[dec_list[ell]]
            else:
                ctext[pos[ell]] = self.alp[dec_list[ell]]
        for j in range(k):
            r += ctext[j]
        return r      
                    
    def compose(self,f,text):
        return self.encode(f.encode(text))

    def __pow__(self,n):
        c = np.array(b)
        for j in range(n-1):
            c = self.A@c + self.b
        return Affine_Hill_Cipher(self.A**n,c,self.alp)
            
    def __str__(self):
        return "Affine Hill Cipher with vector shift " + str(self.b) + " and matrix " + str(self.A)


    def __repr__(self):
        return str(self)

    def __neg__(self):
        return Affine_Hill_Cipher(self.A,-self.b,self.alp)

    def __add__(self,f):
        if not np.array_equal(self.A,f.A):
            raise Exception("Matricies must match for addition.")
        elif self.alp != f.alp:
            raise Exception("Alphabets must match for addition.")
        else:
            return Affine_Hill_Cipher(self.A,self.b+f.b,self.alp)



In [46]:
C = Caesar(3,vowels)
print(C)
print(C.encode(enigma))
encC = C.encode(enigma)
print(C.decode(encC))
print()
A = Affine(3,2,qwerty)
print(A)
encA = A.encode(holmes)
print(encA)
print(A.decode(encA))
print()
F = A.compose(C)
encF = F.encode(bond)
print("F is a composed cipher so cannot be described but is effectively an affine cipher with linear transformation 2N+5")
print(encF)
print(F.decode(encF))
print()
R = Reverse(ambidextrous)
print(R)
encR = R.encode(joker)
print(encR)
print(R.decode(encR))
print()
S = Split(alp)
encS = S.encode(alp)
print(S)
print(encS)
print(S.decode(encS))
print()
A = np.array([[2,1,1],[1,1,1],[1,0,9]])
b = np.array([1,0,2])
H = Affine_Hill_Cipher(A,b,vowels)
encH = H.encode(hamlet)
print(H)
print(encH)
print(H.decode(encH))

Caesar cipher with shift n = 3 over aeiou
[Blutchluy, 1941] Thu Unagmo hos buun crockud!
[Bletchley, 1941] The Enigma has been cracked!

Affine cipher with transfomation a = 3 and b = 2 over qwertyuiopasdfghjklzxcvbnm
[Ug 221Z Zuwos Pgsoog] Cqtmop (gq Yugpqv): "Otomovgusk, mk dous Yugpqv!"
[At 221B Baker Street] Holmes (to Watson): "Elementary, my dear Watson!"

F is a composed cipher so cannot be described but is effectively an affine cipher with linear transformation 2N+5
Qlxvg 007: "Pcqwxv, vog pgussxd!"
Agent 007: "Shaken, not stirred!"

Reverse Cipher over ambidextrous
Why ab axirbma?! Lxd'a pmd s aurlx bn dhsd fscx.
Why so serious?! Let's put a smile on that face.

Split Cipher over abcdefghijklmnopqrstuvwxyz
acegikmoqsuwyzxvtrpnljhfdb
abcdefghijklmnopqrstuvwxyz

Affine Hill Cipher with vector shift [1 0 2] and matrix [[2 1 1]
 [1 1 1]
 [1 0 9]]
[Oct 3, Scine 1] Te bi, ir net ti bu?
[Act 3, Scene 1] To be, or not to be?
